# Michiel's Library Genotyping


In [1]:
## CONFIG ##

# Reload configs on each run.
%load_ext autoreload
%autoreload 2

import csv
from pathlib import Path

import pandas as pd
from IPython.display import display

import config
from config import (BARCODE_MAP, EMPTY_CSV, FASTA_BP, FASTQ_DIR, FILTERED_CSV, FULL_CSV, OUT_DIR,
                    RAW, REMOVED_CSV, SLOTS, STRUCTURAL, VARIANT)
from assembly import decompose

OUT_DIR.mkdir(parents=True, exist_ok=True)

# shown whenever a filter reports what it removed
COLS = ["barcode", "circuit", "read_bp", "region_bp", "parts_bp", "unexplained_bp",
        "coverage", "n_circuit_parts", "valid", "circuit_parts"]

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


In [2]:
def read_length(fastq):
    """Length of the single consensus read in a .final.fastq, or 0 if absent."""
    if not fastq.exists():
        return 0
    with open(fastq) as handle:
        for index, line in enumerate(handle):
            if index == 1:
                return len(line.strip())
    return 0


def get_circuits_with_parts(parts, frame):
    # split on "-" and test membership: "SrpR" is a substring of "tSrpR"
    return frame[frame.circuit_parts.str.split("-").map(set(parts).issubset)]


def circuit_parts(genotype):
    """Called parts with the strand suffix, structural elements and variant numbers removed."""
    parts = []
    for part in genotype.split("-"):
        part = part.rstrip("*")
        if part in ("", "start", "end") or part in STRUCTURAL:
            continue
        parts.append(VARIANT.sub(r"\1", part))
    return parts

## Build the full report


In [3]:
# utf-8-sig: the barcode map carries a BOM
with open(BARCODE_MAP, encoding="utf-8-sig") as handle:
    circuits = {row["Barcode"].strip(): row["Circuit_ID"].strip()
                for row in csv.DictReader(handle)}

called = {}
for line in RAW.read_text().splitlines():
    fields = line.split("\t")
    if len(fields) == 3:
        called[fields[0]] = (fields[1], float(fields[2]))

# every barcode with a FASTQ appears, including the reads that never anchored
barcodes = sorted(path.name.removesuffix(".final.fastq")
                  for path in FASTQ_DIR.glob("barcode*.final.fastq"))

rows = []
for barcode in barcodes:
    genotype, coverage = called.get(barcode, ("", None))
    parts = circuit_parts(genotype) if genotype else []
    parts_bp = sum(FASTA_BP[p.rstrip("*")] for p in genotype.split("-")
                   if p.rstrip("*") in FASTA_BP and p not in ("start", "end")) if genotype else 0
    region_bp = round(parts_bp / coverage) if coverage else 0
    rows.append({
        "barcode": barcode,
        "circuit": circuits.get(barcode, ""),
        "read_bp": read_length(FASTQ_DIR / f"{barcode}.final.fastq"),
        "region_bp": region_bp,
        "parts_bp": parts_bp,
        "coverage": f"{coverage:.4f}" if coverage is not None else "",
        "unexplained_bp": region_bp - parts_bp if region_bp else "",
        "n_circuit_parts": len(parts),
        "circuit_parts": "-".join(parts),
        "valid": decompose(parts, SLOTS) is not None if parts else False,
        "genotype": genotype,
    })

df = pd.DataFrame(rows)
print(f"{len(df)} barcodes, {(df.genotype != '').sum()} genotyped, {df.valid.sum()} valid assemblies")
df.head()

93 barcodes, 87 genotyped, 64 valid assemblies


,barcode,circuit,read_bp,region_bp,parts_bp,coverage,unexplained_bp,n_circuit_parts,circuit_parts,valid,genotype
0,barcode01,circuit_1,35586,3945,3811,0.9660,134,10,AraC-PhlF-pSrpR-pTra-Spacer2-pTac-SrpR-pSrpR-pTra-BM3R1,True,start-start_pad-RiboJ-RBS_S1-AraC-RiboJ10-PhlF-pSrpR-pTr...
1,barcode02,circuit_13,8260,3775,3331,0.8824,444,9,pTac-SrpR-pSrpR-pTra-pBAD-TraRW-pTac-SrpR-Spacer2,False,start-start_pad-pTac-RiboJ10-RBS_S4-SrpR-pSrpR-pTra-pBAD...
2,barcode03,circuit_25,8050,3565,3449,0.9675,116,8,pTac-SrpR-Spacer3-BM3R1-pTac-SrpR-Spacer3-BM3R1,True,start-start_pad-pTac-RiboJ10-RBS_S4-SrpR-Spacer3-RiboJ57...
3,barcode04,circuit_37,7845,3968,3834,0.9662,134,12,AraC-PhlF-pSrpR-pTra-pBAD-Spacer1-pTac-SrpR-pSrpR-pTra-p...,True,start-start_pad-RiboJ-RBS_S1-AraC-RiboJ10-PhlF-pSrpR-pTr...
4,barcode05,circuit_49,9084,4608,4473,0.9706,135,12,pTac-SrpR-pSrpR-pTra-pBAD-Spacer2-AraC-PhlF-pSrpR-pTra-p...,True,start-start_pad-pTac-RiboJ10-RBS_S4-SrpR-pSrpR-pTra-pBAD...


In [4]:
df.to_csv(FULL_CSV, index=False)

print(f"wrote {FULL_CSV}")
print(f"{len(df)} barcodes total")

wrote /home/ado/mscr/dev/seq/out/michiel_lib_genotype_full.csv
93 barcodes total


## Progressive filtering


In [5]:
# Filters run in order: no-genotype rows first, their unexplained_bp is blank
# The RPU reference is a calibration construct, not a circuit
filtered = df.copy()

removed = filtered[filtered.circuit == "RPU"]
filtered = filtered[filtered.circuit != "RPU"]

print(f"RPU reference: removed {len(removed)}, {len(filtered)} remain")
display(removed[COLS])

RPU reference: removed 1, 92 remain


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
71,barcode72,RPU,7793,3307,3172,135,0.9592,10,True,pTac-SrpR-pSrpR-pTra-Spacer2-AraC-PhlF-pSrpR-pTra-Spacer1


In [6]:
# mseg returned nothing: the read never spanned both anchors
removed = filtered[filtered.genotype == ""]
filtered = filtered[filtered.genotype != ""]

print(f"no genotype: removed {len(removed)}, {len(filtered)} remain")
display(removed[COLS])

no genotype: removed 6, 86 remain


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
19,barcode20,circuit_39,8979,0,0,,,0,False,
27,barcode28,circuit_40,7839,0,0,,,0,False,
30,barcode31,circuit_76,5883,0,0,,,0,False,
41,barcode42,circuit_18,9066,0,0,,,0,False,
77,barcode78,circuit_70,5511,0,0,,,0,False,
91,barcode94,circuit_72,9432,0,0,,,0,False,


In [7]:
# Read far longer than the classified region
gap = filtered.read_bp - filtered.region_bp

removed = filtered[gap > 5_000]
filtered = filtered[~(gap > 5_000)]

print(f"read_bp - region_bp > 5,000: removed {len(removed)}, {len(filtered)} remain")
display(removed[COLS])

read_bp - region_bp > 5,000: removed 5, 81 remain


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
0,barcode01,circuit_1,35586,3945,3811,134,0.9660,10,True,AraC-PhlF-pSrpR-pTra-Spacer2-pTac-SrpR-pSrpR-pTra-BM3R1
17,barcode18,circuit_15,31303,3844,3638,206,0.9464,9,True,pTac-SrpR-Spacer3-Spacer1-AraC-PhlF-pSrpR-pTra-BM3R1
53,barcode54,circuit_67,14845,3252,3080,172,0.9470,8,False,pTac-SrpR-pSrpR-pTra-pBAD-BM3R1-pTac-SrpR
84,barcode86,circuit_71,30631,13466,5669,7797,0.4210,12,False,Spacer2-AraC-PhlF-pTet-pTac-tSrpR-BM3R1-Spacer2-AraC-Phl...
92,barcode95,circuit_84,27967,4394,4181,213,0.9515,8,True,pTac-SrpR-Spacer3-BM3R1-AraC-PhlF-Spacer3-BM3R1


In [8]:
# Sequence between the anchors that no part accounts for
unexplained = pd.to_numeric(filtered.unexplained_bp, errors="coerce")

removed = filtered[unexplained > 200]
filtered = filtered[~(unexplained > 200)]

print(f"unexplained_bp > 200: removed {len(removed)}, {len(filtered)} remain")
display(removed[COLS])

unexplained_bp > 200: removed 13, 68 remain


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
1,barcode02,circuit_13,8260,3775,3331,444,0.8824,9,False,pTac-SrpR-pSrpR-pTra-pBAD-TraRW-pTac-SrpR-Spacer2
5,barcode06,circuit_61,36384,32487,20389,12098,0.6276,41,False,AraC-PhlF-pSrpR-pTra-pBAD-AraC-PhlF-pSrpR-pTra-pBAD-BM3R...
6,barcode07,circuit_73,9230,4704,4428,276,0.9414,8,True,AraC-PhlF-Spacer3-Spacer1-AraC-PhlF-Spacer3-BM3R1
12,barcode13,circuit_50,7469,2916,2532,384,0.8683,6,False,AraC-PhlF-pSrpR-pTra-pBAD-pTac
13,barcode14,circuit_62,19988,15456,6306,9150,0.4080,24,False,pTac-SrpR-Spacer3-Spacer2-pTac-SrpR-Spacer3-Spacer2-pTac...
21,barcode22,circuit_63,8786,4299,3404,895,0.7919,8,False,AraC-PhlF-Spacer3-SrpR-pSrpR-pTra-pBAD-Spacer2
29,barcode30,circuit_64,18642,14164,1000,13164,0.0706,0,False,
35,barcode36,circuit_41,7717,3232,3009,223,0.9310,8,False,PhlF-Spacer3-Spacer2-pTac-SrpR-pSrpR-pTra-TraRW
47,barcode48,circuit_90,9312,4827,3354,1473,0.6948,10,True,AraC-PhlF-pTet-pTet-tPhlF-pTac-SrpR-pSrpR-pTra-Spacer1
70,barcode71,circuit_81,19124,15399,6232,9167,0.4047,23,False,pTac-SrpR-Spacer3-Spacer2-pTac-SrpR-Spacer3-Spacer1-pTac...


In [9]:
filtered.to_csv(FILTERED_CSV, index=False)

print(f"wrote {FILTERED_CSV}")
print(f"{len(filtered)} barcodes survived filtering")

wrote /home/ado/mscr/dev/seq/out/michiel_lib_genotype_filtered.csv
68 barcodes survived filtering


In [10]:
# mseg returned nothing for these, so there is no genotype for a filter to have rejected
empty = df[(df.circuit != "RPU") & (df.genotype == "")]
empty.to_csv(EMPTY_CSV, index=False)

print(f"wrote {EMPTY_CSV}")
print(f"{len(empty)} circuit barcodes never assembled across the junction")

wrote /home/ado/mscr/dev/seq/out/michiel_lib_genotype_empty.csv
6 circuit barcodes never assembled across the junction


In [11]:
# Genotyped circuits a coverage filter rejected
# The 92 circuit barcodes partition: empty + removed + filtered
removed = df[(df.circuit != "RPU") & (df.genotype != "") & ~df.barcode.isin(filtered.barcode)]
removed.to_csv(REMOVED_CSV, index=False)

print(f"wrote {REMOVED_CSV}")
print(f"{len(removed)} genotyped circuit barcodes were filtered out")
assert len(empty) + len(removed) + len(filtered) == int((df.circuit != "RPU").sum())

wrote /home/ado/mscr/dev/seq/out/michiel_lib_genotype_removed.csv
18 genotyped circuit barcodes were filtered out


## Valid and invalid assemblies


In [12]:
# `valid`: a legal a-b-c-a-b-c assembly, independent of coverage
valid = filtered[filtered.valid]
invalid = filtered[~filtered.valid]

print(f"{len(valid)} valid, {len(invalid)} invalid, of {len(filtered)} circuits")
display(invalid[COLS])

58 valid, 10 invalid, of 68 circuits


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
10,barcode11,circuit_26,7161,2674,2505,169,0.9368,6,False,AraC-PhlF-pSrpR-pTra-pBAD-Spacer2
15,barcode16,circuit_86,8546,4056,3920,136,0.9665,10,False,pTac-AraC-PhlF-Spacer3-TraRW-pTac-SrpR-pSrpR-pTra-Spacer1
36,barcode37,circuit_53,7101,2616,2501,115,0.9560,6,False,AraC-PhlF-pSrpR-pTra-pBAD-Spacer2
45,barcode46,circuit_66,7828,3342,3224,118,0.9647,8,False,AraC-PhlF-Spacer3-pTet-BM3R1-pBM3R1-pBAD-Spacer2
51,barcode52,circuit_43,7509,3024,2897,127,0.9580,5,False,AraC-PhlF-Spacer3-BM3R1-pTac
59,barcode60,circuit_44,6490,2001,1904,97,0.9516,4,False,pTac-SrpR-Spacer3-BM3R1
62,barcode63,circuit_80,8753,4268,4143,125,0.9707,14,False,pTac-SrpR-pSrpR-pTra-pBAD-BM3R1-pTac-SrpR-Spacer3-Spacer...
64,barcode65,circuit_9,6670,2186,1997,189,0.9135,4,False,AraC-PhlF-Spacer3-Spacer1
82,barcode84,circuit_47,8328,3843,3723,120,0.9688,6,False,AraC-PhlF-Spacer3-BM3R1-pBAD-BM3R1
88,barcode91,circuit_36,6631,2145,2023,122,0.9431,7,False,pTac-SrpR-Spacer3-Spacer1-PhlF-Spacer3-Spacer2


In [13]:
display(valid[COLS])

,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
2,barcode03,circuit_25,8050,3565,3449,116,0.9675,8,True,pTac-SrpR-Spacer3-BM3R1-pTac-SrpR-Spacer3-BM3R1
3,barcode04,circuit_37,7845,3968,3834,134,0.9662,12,True,AraC-PhlF-pSrpR-pTra-pBAD-Spacer1-pTac-SrpR-pSrpR-pTra-p...
4,barcode05,circuit_49,9084,4608,4473,135,0.9706,12,True,pTac-SrpR-pSrpR-pTra-pBAD-Spacer2-AraC-PhlF-pSrpR-pTra-p...
7,barcode08,circuit_85,8500,4014,3898,116,0.9711,12,True,pTac-SrpR-pTet-pTac-tSrpR-BM3R1-BM3R1-pTac-SrpR-pSrpR-pT...
8,barcode09,circuit_2,8439,3954,3811,143,0.9638,10,True,AraC-PhlF-pSrpR-pTra-Spacer2-pTac-SrpR-pSrpR-pTra-BM3R1
9,barcode10,circuit_14,8666,4180,4046,134,0.9679,10,True,AraC-PhlF-pSrpR-pTra-pBAD-Spacer1-pTac-SrpR-Spacer3-BM3R1
11,barcode12,circuit_38,8340,3849,3715,134,0.9652,9,True,AraC-PhlF-pSrpR-pTra-Spacer1-pTac-SrpR-Spacer3-BM3R1
14,barcode15,circuit_74,10004,5519,5384,135,0.9755,11,True,pTac-SrpR-pTet-pTac-tSrpR-BM3R1-BM3R1-AraC-PhlF-Spacer3-...
16,barcode17,circuit_3,9012,4527,4392,135,0.9702,8,True,AraC-PhlF-Spacer3-TraRW-pTac-SrpR-Spacer3-BM3R1
18,barcode19,circuit_27,8500,4001,3859,142,0.9646,9,True,pTac-SrpR-Spacer3-pTet-tPhlF-AraC-PhlF-Spacer3-BM3R1


## Duplicate genotypes


In [14]:
# Distinct barcodes calling the same genotype; reported only
duplicated = valid[valid.duplicated("genotype", keep=False)]

print(f"valid: {duplicated.genotype.nunique()} genotypes shared by {len(duplicated)} barcodes")
for genotype, group in duplicated.groupby("genotype", sort=False):
    display(group[COLS])

valid: 4 genotypes shared by 8 barcodes


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
23,barcode24,circuit_87,8523,4037,3885,152,0.9623,9,True,AraC-PhlF-pSrpR-pTra-Spacer1-AraC-PhlF-Spacer3-Spacer2
39,barcode40,circuit_89,8523,4038,3885,153,0.9621,9,True,AraC-PhlF-pSrpR-pTra-Spacer1-AraC-PhlF-Spacer3-Spacer2


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
24,barcode25,circuit_4,8026,3542,3407,135,0.9619,10,True,AraC-PhlF-pSrpR-pTra-pBAD-Spacer1-pTac-SrpR-Spacer3-Spacer2
38,barcode39,circuit_77,8027,3542,3407,135,0.9619,10,True,AraC-PhlF-pSrpR-pTra-pBAD-Spacer1-pTac-SrpR-Spacer3-Spacer2


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
25,barcode26,circuit_16,10132,5647,5494,153,0.9729,10,True,AraC-PhlF-pSrpR-pTra-pBAD-BM3R1-AraC-PhlF-Spacer3-BM3R1
58,barcode59,circuit_32,10133,5646,5494,152,0.9731,10,True,AraC-PhlF-pSrpR-pTra-pBAD-BM3R1-AraC-PhlF-Spacer3-BM3R1


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
55,barcode56,circuit_91,8523,4037,3885,152,0.9623,9,True,AraC-PhlF-Spacer3-Spacer2-AraC-PhlF-pSrpR-pTra-Spacer1
63,barcode64,circuit_92,8523,4038,3885,153,0.9621,9,True,AraC-PhlF-Spacer3-Spacer2-AraC-PhlF-pSrpR-pTra-Spacer1


In [15]:
duplicated = invalid[invalid.duplicated("genotype", keep=False)]

print(f"invalid: {duplicated.genotype.nunique()} genotypes shared by {len(duplicated)} barcodes")
for genotype, group in duplicated.groupby("genotype", sort=False):
    display(group[COLS])

invalid: 0 genotypes shared by 0 barcodes


In [16]:
# Do barcodes calling one genotype get the same phenotype?
CLASSIFY_CSV = Path("../nb/out/analyse_pr_classify.csv")

calls = pd.read_csv(CLASSIFY_CSV).set_index("Group")["logical_function_cello"]

pairs = filtered[filtered.duplicated("genotype", keep=False)].copy()
pairs["logical_function_kl"] = [calls.get(f"Circuit {int(circuit.split('_')[1])}")
                                for circuit in pairs.circuit]
disagree = pairs.groupby("genotype").filter(
    lambda group: group.logical_function_kl.nunique(dropna=False) > 1)

print(f"{disagree.genotype.nunique()} of {pairs.genotype.nunique()} duplicate genotypes "
      f"disagree on phenotype (KL)")
display(disagree[["barcode", "circuit", "logical_function_kl", "circuit_parts"]])

0 of 4 duplicate genotypes disagree on phenotype (KL)


,barcode,circuit,logical_function_kl,circuit_parts


In [17]:
# Do barcodes calling one genotype get the same phenotype?
CLASSIFY_CSV = Path("../nb/out/analyse_pr_classify.csv")

calls = pd.read_csv(CLASSIFY_CSV).set_index("Group")["logical_function_cello"]

pairs = filtered[filtered.duplicated("genotype", keep=False)].copy()
pairs["logical_function_cello"] = [calls.get(f"Circuit {int(circuit.split('_')[1])}")
                                for circuit in pairs.circuit]
disagree = pairs.groupby("genotype").filter(
    lambda group: group.logical_function_cello.nunique(dropna=False) > 1)

print(f"{disagree.genotype.nunique()} of {pairs.genotype.nunique()} duplicate genotypes "
      f"disagree on phenotype (Cello Score)")
display(disagree[["barcode", "circuit", "logical_function_cello", "circuit_parts"]])

0 of 4 duplicate genotypes disagree on phenotype (Cello Score)


,barcode,circuit,logical_function_cello,circuit_parts


## Summary


In [18]:
# "unique" collapses barcodes that called the same genotype.
# percentages are of the circuit barcodes, every barcode but the RPU reference.
n_circuits = int((df.circuit != "RPU").sum())

n_filtered = len(filtered)
n_unique = filtered.genotype.nunique()
n_valid = len(valid)
n_unique_valid = valid.genotype.nunique()

print(f"circuit barcodes       {n_circuits:3d}")
print(f"total circuits         {n_filtered:3d}   ({n_filtered / n_circuits * 100:.1f}%)")
print(f"unique circuits        {n_unique:3d}   ({n_unique / n_circuits * 100:.1f}%)")
print(f"total valid circuits   {n_valid:3d}   ({n_valid / n_circuits * 100:.1f}%)")
print(f"unique valid circuits  {n_unique_valid:3d}   ({n_unique_valid / n_circuits * 100:.1f}%)")

print(f"")
n_total_lib_circuits = 14_400
print(f"unique valid circuits {n_unique_valid:3d} out of total ({n_unique_valid / n_total_lib_circuits * 100:.1f}%)")

circuit barcodes        92
total circuits          68   (73.9%)
unique circuits         64   (69.6%)
total valid circuits    58   (63.0%)
unique valid circuits   54   (58.7%)

unique valid circuits  54 out of total (0.4%)


## Part Composition

### Filtered out

In [19]:
## CONFIG ##

# From config's vocabulary
SPACERS = sorted(part for part in config.PART_BP if part.startswith("Spacer"))

for parts in [[spacer] for spacer in SPACERS] + [SPACERS]:
    matching = get_circuits_with_parts(parts, removed)
    print(f"{' + '.join(parts):30s} {len(matching):3d}"
          f"   ({len(matching) / len(removed) * 100:.1f}%)")

display(get_circuits_with_parts(SPACERS, removed)[COLS])

Spacer1                          7   (38.9%)
Spacer2                          7   (38.9%)
Spacer3                          9   (50.0%)
Spacer1 + Spacer2 + Spacer3      2   (11.1%)


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
13,barcode14,circuit_62,19988,15456,6306,9150,0.4080,24,False,pTac-SrpR-Spacer3-Spacer2-pTac-SrpR-Spacer3-Spacer2-pTac...
70,barcode71,circuit_81,19124,15399,6232,9167,0.4047,23,False,pTac-SrpR-Spacer3-Spacer2-pTac-SrpR-Spacer3-Spacer1-pTac...


In [20]:
## CONFIG ##

# tSrpR and tPhlF, both repressor-gated; tEND is ours and is not a library part
INDUCIBLE_TERMINATORS = sorted(part for part in config.PART_BP if part.startswith("t"))

for parts in [[terminator] for terminator in INDUCIBLE_TERMINATORS] + [INDUCIBLE_TERMINATORS]:
    matching = get_circuits_with_parts(parts, removed)
    print(f"{' + '.join(parts):30s} {len(matching):3d}"
          f"   ({len(matching) / len(removed) * 100:.1f}%)")

display(get_circuits_with_parts(INDUCIBLE_TERMINATORS, removed)[COLS])

tPhlF                            1   (5.6%)
tSrpR                            1   (5.6%)
tPhlF + tSrpR                    0   (0.0%)


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts


In [21]:
## CONFIG ##

_parts = sorted(part for part in config.PART_BP if part.startswith("pTet"))

for parts in [[part] for part in _parts] + [_parts]:
    matching = get_circuits_with_parts(parts, removed)
    print(f"{' + '.join(parts):30s} {len(matching):3d}"
          f"   ({len(matching) / len(removed) * 100:.1f}%)")

display(get_circuits_with_parts(_parts, removed)[COLS])

pTet                             3   (16.7%)
pTet                             3   (16.7%)


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
47,barcode48,circuit_90,9312,4827,3354,1473,0.6948,10,True,AraC-PhlF-pTet-pTet-tPhlF-pTac-SrpR-pSrpR-pTra-Spacer1
83,barcode85,circuit_59,28712,24216,13544,10672,0.5593,30,False,AraC-PhlF-Spacer3-Spacer1-AraC-PhlF-Spacer3-BM3R1-AraC-P...
84,barcode86,circuit_71,30631,13466,5669,7797,0.4210,12,False,Spacer2-AraC-PhlF-pTet-pTac-tSrpR-BM3R1-Spacer2-AraC-Phl...


### Invalid

In [22]:
## CONFIG ##

# From config's vocabulary
SPACERS = sorted(part for part in config.PART_BP if part.startswith("Spacer"))

for parts in [[spacer] for spacer in SPACERS] + [SPACERS]:
    matching = get_circuits_with_parts(parts, invalid)
    print(f"{' + '.join(parts):30s} {len(matching):3d}"
          f"   ({len(matching) / len(invalid) * 100:.1f}%)")

display(get_circuits_with_parts(SPACERS, invalid)[COLS])

Spacer1                          4   (40.0%)
Spacer2                          4   (40.0%)
Spacer3                          8   (80.0%)
Spacer1 + Spacer2 + Spacer3      1   (10.0%)


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
88,barcode91,circuit_36,6631,2145,2023,122,0.9431,7,False,pTac-SrpR-Spacer3-Spacer1-PhlF-Spacer3-Spacer2


In [23]:
## CONFIG ##

# tSrpR and tPhlF, both repressor-gated; tEND is ours and is not a library part
INDUCIBLE_TERMINATORS = sorted(part for part in config.PART_BP if part.startswith("t"))

for parts in [[terminator] for terminator in INDUCIBLE_TERMINATORS] + [INDUCIBLE_TERMINATORS]:
    matching = get_circuits_with_parts(parts, invalid)
    print(f"{' + '.join(parts):30s} {len(matching):3d}"
          f"   ({len(matching) / len(invalid) * 100:.1f}%)")

display(get_circuits_with_parts(INDUCIBLE_TERMINATORS, invalid)[COLS])

tPhlF                            0   (0.0%)
tSrpR                            0   (0.0%)
tPhlF + tSrpR                    0   (0.0%)


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts


In [24]:
## CONFIG ##

_parts = sorted(part for part in config.PART_BP if part.startswith("pTet"))

for parts in [[part] for part in _parts] + [_parts]:
    matching = get_circuits_with_parts(parts, invalid)
    print(f"{' + '.join(parts):30s} {len(matching):3d}"
          f"   ({len(matching) / len(invalid) * 100:.1f}%)")

display(get_circuits_with_parts(_parts, invalid)[COLS])

pTet                             1   (10.0%)
pTet                             1   (10.0%)


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
45,barcode46,circuit_66,7828,3342,3224,118,0.9647,8,False,AraC-PhlF-Spacer3-pTet-BM3R1-pBM3R1-pBAD-Spacer2


### Valid

In [25]:
## CONFIG ##

# From config's vocabulary
SPACERS = sorted(part for part in config.PART_BP if part.startswith("Spacer"))

for parts in [[spacer] for spacer in SPACERS] + [SPACERS]:
    matching = get_circuits_with_parts(parts, valid)
    print(f"{' + '.join(parts):30s} {len(matching):3d}"
          f"   ({len(matching) / len(valid) * 100:.1f}%)")

display(get_circuits_with_parts(SPACERS, valid)[COLS])

Spacer1                         29   (50.0%)


Spacer2                         25   (43.1%)
Spacer3                         43   (74.1%)
Spacer1 + Spacer2 + Spacer3      8   (13.8%)


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
20,barcode21,circuit_51,7666,3212,3076,136,0.9576,9,True,pTac-SrpR-pSrpR-pTra-Spacer1-AraC-PhlF-Spacer3-Spacer2
23,barcode24,circuit_87,8523,4037,3885,152,0.9623,9,True,AraC-PhlF-pSrpR-pTra-Spacer1-AraC-PhlF-Spacer3-Spacer2
24,barcode25,circuit_4,8026,3542,3407,135,0.9619,10,True,AraC-PhlF-pSrpR-pTra-pBAD-Spacer1-pTac-SrpR-Spacer3-Spacer2
38,barcode39,circuit_77,8027,3542,3407,135,0.9619,10,True,AraC-PhlF-pSrpR-pTra-pBAD-Spacer1-pTac-SrpR-Spacer3-Spacer2
39,barcode40,circuit_89,8523,4038,3885,153,0.9621,9,True,AraC-PhlF-pSrpR-pTra-Spacer1-AraC-PhlF-Spacer3-Spacer2
55,barcode56,circuit_91,8523,4037,3885,152,0.9623,9,True,AraC-PhlF-Spacer3-Spacer2-AraC-PhlF-pSrpR-pTra-Spacer1
63,barcode64,circuit_92,8523,4038,3885,153,0.9621,9,True,AraC-PhlF-Spacer3-Spacer2-AraC-PhlF-pSrpR-pTra-Spacer1
78,barcode79,circuit_82,8854,4369,4216,153,0.9650,10,True,AraC-PhlF-Spacer3-Spacer2-AraC-PhlF-pSrpR-pTra-pBAD-Spacer1


In [26]:
## CONFIG ##

# tSrpR and tPhlF, both repressor-gated; tEND is ours and is not a library part
INDUCIBLE_TERMINATORS = sorted(part for part in config.PART_BP if part.startswith("t"))

for parts in [[terminator] for terminator in INDUCIBLE_TERMINATORS] + [INDUCIBLE_TERMINATORS]:
    matching = get_circuits_with_parts(parts, valid)
    print(f"{' + '.join(parts):30s} {len(matching):3d}"
          f"   ({len(matching) / len(valid) * 100:.1f}%)")

display(get_circuits_with_parts(INDUCIBLE_TERMINATORS, valid)[COLS])

tPhlF                            3   (5.2%)
tSrpR                            3   (5.2%)
tPhlF + tSrpR                    0   (0.0%)


,barcode,circuit,read_bp,region_bp,parts_bp,unexplained_bp,coverage,n_circuit_parts,valid,circuit_parts
